# **Build a LLM From Scratch - Sebastian Raschka**

#### Verify PyTorch and CUDA

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import multiprocessing
print(torch.__version__)
print(torch.cuda.is_available())
multiprocessing.cpu_count()

2.11.0+cu128
True


12

#### Configuration

In [ ]:
import os

torch.manual_seed(1009)

VOCAB_SIZE = 45098
CONTEXT_LEN = 516
EMB_DIM = 1024
N_HEADS = 16
N_LAYERS = 24
DROPOUT = 0.1
QKV_BIAS = False
TRAIN_RATIO = 0.9

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DTYPE = torch.bfloat16
CORES = 10
SAMPLE_BYTES = 30 * 1024 * 1024
TOKENIZER_BYTES = 500 * 1024 * 1024
PRETRAINING_GB = 10 * 1024 * 1024 * 1024
SAMPLE_FILE = "academic_sample.txt"
TOKENIZER_FILE = "academic_tokenizing.txt"
TOKENIZER_CONFIG = "tokenizer_config.txt"
PRETRAINING_FILE = "academic_pretraining.txt"
DATA = [TOKENIZER_FILE, TOKENIZER_BYTES]

#### Load and store data

In [ ]:
os.environ["HF_TOKEN"] = ""

from datasets import load_dataset
print("Connecting to Hugging Face dataset stream...")
dataset = load_dataset(
    "allenai/peS2o", split="train", streaming=True, trust_remote_code=True
)

bytes_written = 0
line_count = 0
print(f"Writing to file {DATA[0]}...")

with open(DATA[0], "w", encoding="utf-8") as f:
    for r in dataset:
        raw = r.get("text", "").strip()

        if raw:
            text = " ".join(raw.splitlines())
            to_write = text + "\n"
            f.write(to_write)

            bytes_written += len(to_write.encode("utf-8"))
            line_count += 1

            if line_count % 15000 == 0:
                mb_written = bytes_written / (1024 * 1024)
                print(f"Written: {mb_written:.2f} MB ({line_count} documents)")

            if bytes_written >= DATA[1]:
                print("Max bytes reached. Disconnecting stream")
                break

print(f"Done.\nFile saved as {DATA[0]}, {bytes_written / (1024 * 1024):.2f} MB")

Connecting to Hugging Face dataset stream...
Writing to file academic_tokenizing.txt...
Written: 19.00 MB (15000 documents)
Written: 37.94 MB (30000 documents)
Written: 56.79 MB (45000 documents)
Written: 75.71 MB (60000 documents)
Written: 94.73 MB (75000 documents)
Written: 113.77 MB (90000 documents)
Written: 132.61 MB (105000 documents)
Written: 151.52 MB (120000 documents)
Written: 170.38 MB (135000 documents)
Written: 189.32 MB (150000 documents)
Written: 208.30 MB (165000 documents)
Written: 227.27 MB (180000 documents)
Written: 246.17 MB (195000 documents)
Written: 265.08 MB (210000 documents)
Written: 283.94 MB (225000 documents)
Written: 302.75 MB (240000 documents)
Written: 321.72 MB (255000 documents)
Written: 340.61 MB (270000 documents)
Written: 359.52 MB (285000 documents)
Written: 378.41 MB (300000 documents)
Written: 397.41 MB (315000 documents)
Written: 416.41 MB (330000 documents)
Written: 435.36 MB (345000 documents)
Written: 454.26 MB (360000 documents)
Written: 47

In [11]:
import pathlib


def load_tokenizer_data():
    if os.path.isfile(TOKENIZER_FILE):
        with open(TOKENIZER_FILE, "r", encoding="utf-8") as f:
            raw = f.readlines()
        return [l.rstrip() for l in raw]
    else:
        print(f"File {pathlib.Path(TOKENIZER_FILE).resolve()} not found")


def load_sample_data():
    if os.path.isfile(SAMPLE_FILE):
        with open(SAMPLE_FILE, "r", encoding="utf-8") as f:
            return f.readlines()
    else:
        print(f"File {pathlib.Path(SAMPLE_FILE).resolve()} not found")

## Tokenizer

In [17]:
from collections import defaultdict
from multiprocessing import Pool, cpu_count
from functools import lru_cache
import heapq
import ast, re

TOKENIZER_DATA = load_tokenizer_data()

pattern = re.compile(
    r'<\|endoftext\|>|<\|unk\|>|--|\s*[A-Za-z0-9]+|[\(\)\[\],._?!"\'—%&$€£-]'
)


def pre_tokenize(line):
    words = pattern.findall(line)
    return [w.replace(" ", "G̃") for w in words]


def _worker(lines):
    local = defaultdict(int)
    for line in lines:
        for w in pre_tokenize(line):
            local[w] += 1
    return local


def compute_word_freqs(data):
    print("Starting pre-tokenization...")

    chunk_size = (len(data) + CORES - 1) // CORES
    chunks_data = [data[i : i + chunk_size] for i in range(0, len(data), chunk_size)]

    with Pool(cpu_count()) as pool:
        results = pool.map(_worker, chunks_data)

    print("Chunks done")

    word_freqs = defaultdict(int)
    for r in results:
        for k, v in r.items():
            word_freqs[k] += v

    print("Pre-tokenization done")

    return word_freqs


def base_vocab(word_freqs):
    alphabet = []
    for w in word_freqs.keys():
        for l in w:
            if l not in alphabet:
                alphabet.append(l)
    alphabet.sort()
    print("Base vocabulary initialized, length:", len(alphabet) + 2)
    return ["<|endoftext|>", "<|unk|>"] + alphabet.copy()


def init_structures(word_freqs):
    splits = {w: list(w) for w in word_freqs}

    pair_freqs = defaultdict(int)
    pair_positions = defaultdict(set)

    for w, f in word_freqs.items():
        tokens = splits[w]

        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pair_freqs[pair] += f
            pair_positions[pair].add((w, i))

    heap = [(-f, pair) for pair, f in pair_freqs.items()]
    heapq.heapify(heap)

    return splits, pair_freqs, pair_positions, heap


def merge(pair, splits, pair_freqs, pair_positions, heap, word_freqs):
    newi = "".join([*pair])
    affected = list(pair_positions[pair])
    pair_positions.pop(pair, None)

    for w, i in affected:
        tokens = splits[w]
        if i >= len(tokens) - 1 or tokens[i] != pair[0] or tokens[i + 1] != pair[1]:
            continue

        freq = word_freqs[w]

        if i > 0:
            l = (tokens[i - 1], tokens[i])
            pair_freqs[l] -= freq
            pair_positions[l].discard((w, i - 1))

        if i + 2 < len(tokens):
            r = (tokens[i + 1], tokens[i + 2])
            pair_freqs[r] -= freq
            pair_positions[r].discard((w, i + 1))

        tokens[i : i + 2] = [newi]

        if i > 0:
            lnew = (tokens[i - 1], newi)
            pair_freqs[lnew] += freq
            pair_positions[lnew].add((w, i - 1))
            heapq.heappush(heap, (-pair_freqs[lnew], lnew))

        if i < len(tokens) - 1:
            rnew = (newi, tokens[i + 1])
            pair_freqs[rnew] += freq
            pair_positions[rnew].add((w, i))
            heapq.heappush(heap, (-pair_freqs[rnew], rnew))


def create_vocab(data, vocab_size):
    word_freqs = compute_word_freqs(data)
    vocab = base_vocab(word_freqs)

    splits, pair_freqs, pair_positions, heap = init_structures(word_freqs)
    merges = {}

    while len(vocab) < vocab_size and heap:
        freq, pair = heapq.heappop(heap)
        freq = -freq

        if pair_freqs[pair] != freq or freq == 0:
            continue

        merge(pair, splits, pair_freqs, pair_positions, heap, word_freqs)

        newi = "".join([*pair])
        merges[pair] = newi
        vocab.append(newi)

        if len(vocab) % 1000 == 0:
            print("Vocab size:", len(vocab))

    token_to_id = {t: i for i, t in enumerate(vocab)}
    id_to_token = {i: t for t, i in token_to_id.items()}

    return token_to_id, id_to_token, vocab, merges


class BPETokenizer:
    def __init__(self, data=TOKENIZER_DATA, vocab_size=VOCAB_SIZE, new_vocab=False):
        if new_vocab:
            config = create_vocab(data, vocab_size)
        else:
            with open("tokenizer_config.txt", "r", encoding="utf-8") as f:
                config = ast.literal_eval(f.read())

        self.token_to_id, self.id_to_token, self.vocab, self.merges = config
        self.merge_ranks = {pair: i for i, pair in enumerate(self.merges)}

        if new_vocab and input("Save tokenizer configuration? [y/N]: ") == "y":
            settings = [self.token_to_id, self.id_to_token, self.vocab, self.merges]
            with open(TOKENIZER_CONFIG, "w", encoding="utf-8") as f:
                f.write(f"{settings}")

    def tokenize_word(self, word):
        tokens = list(word)
        if word in {"<|endoftext|>", "<|unk|>"} or len(tokens) <= 1:
            return [word]

        while True:
            best_rank = None
            best_idx = None

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                rank = self.merge_ranks.get(pair)
                if rank is not None:
                    if best_rank is None or rank < best_rank:
                        best_rank = rank
                        best_idx = i

            if best_rank is None:
                break
            tokens[best_idx : best_idx + 2] = [tokens[best_idx] + tokens[best_idx + 1]]

        return tokens

    @lru_cache(maxsize=200_000)
    def tokenize_word_cached(self, word):
        return tuple(self.tokenize_word(word))

    def tokenize(self, text):
        words = pre_tokenize(text)
        tokens = []
        for w in words:
            tokens.extend(self.tokenize_word_cached(w))

        return list(tokens)

    def pad1d(self, tensors):
        maxi = max(t.shape[0] for t in tensors)

        if all(t.shape[0] == maxi for t in tensors):
            return tensors

        return [F.pad(t, (0, maxi - t.shape[0])) for t in tensors]

    def encode(self, *texts):
        if len(texts) == 0:
            raise TypeError(
                "At least 2 argument must be given to BPETokenizer.encode()"
            )

        tensors = [
            torch.tensor([self.token_to_id.get(t, 1) for t in self.tokenize(text)])
            for text in texts
        ]

        return tensors[0] if len(tensors) == 1 else self.pad1d(tensors)

    def decode(self, *tensors):
        if len(tensors) == 0:
            raise TypeError(
                "At least 2 argument must be given to BPETokenizer.decode()"
            )

        texts = [
            "".join([self.id_to_token[x] for x in t.tolist()]).replace("G̃", " ")
            for t in tensors
        ]

        return texts[0] if len(texts) == 1 else tensors

In [13]:
if __name__ == "__main__":
    tokenizeri = BPETokenizer(new_vocab=True)

    print(
        tokenizeri.tokenize(
            "The ([dominant]) sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train."
        )
    )
    ids = tokenizeri.encode(
        "The ([dominant]) sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train."
    )
    print(ids)
    print(len(ids))
    print(tokenizeri.decode(ids))
    print(len(tokenizeri.vocab))

Starting pre-tokenization...
Chunks done
Pre-tokenization done
Base vocabulary initialized, length: 98
Vocab size: 1000
Vocab size: 2000
Vocab size: 3000
Vocab size: 4000
Vocab size: 5000
Vocab size: 6000
Vocab size: 7000
Vocab size: 8000
Vocab size: 9000
Vocab size: 10000
Vocab size: 11000
Vocab size: 12000
Vocab size: 13000
Vocab size: 14000
Vocab size: 15000
Vocab size: 16000
Vocab size: 17000
Vocab size: 18000
Vocab size: 19000
Vocab size: 20000
Vocab size: 21000
Vocab size: 22000
Vocab size: 23000
Vocab size: 24000
Vocab size: 25000
Vocab size: 26000
Vocab size: 27000
Vocab size: 28000
Vocab size: 29000
Vocab size: 30000
Vocab size: 31000
Vocab size: 32000
Vocab size: 33000
Vocab size: 34000
Vocab size: 35000
Vocab size: 36000
Vocab size: 37000
Vocab size: 38000
Vocab size: 39000
Vocab size: 40000
Vocab size: 41000
Vocab size: 42000
Vocab size: 43000
Vocab size: 44000
Vocab size: 45000
['The', '(', '[', 'domina', 'n', 't', ']', ')', 'G̃sequence', 'G̃t', 'rans', 'duction', 'G̃mo', 

## Dataset and dataloader

In [41]:
class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        tokens = tokenizer.encode(txt)
        windows = tokens.unfold(0, max_length + 1, stride)

        self.input_ids = windows[:, :-1]
        self.target_ids = windows[:, 1:]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(
    txt,
    tokenizer,
    batch_size=128,
    max_length=CONTEXT_LEN,
    stride=CONTEXT_LEN,
    shuffle=True,
    drop_last=True,
    num_workers=CORES,
):
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader

In [18]:
if __name__ == "__main__":
    tokenizeri = BPETokenizer()
    dataloader = create_dataloader(
        "<|endoftext|>".join(TOKENIZER_DATA[:5]),
        tokenizer=tokenizeri,
        batch_size=8,
        max_length=4,
        stride=4,
        shuffle=False,
    )
    data_iter = iter(dataloader)
    inputs, targets = next(data_iter)
    print("Inputs:\n", inputs)
    print("\nTargets:\n", targets)

Inputs:
 tensor([[   52,    44,    62,    69],
        [   72,    74,    13,    74],
        [   59,    72,    67,  1092],
        [  117,    68,    59,    73],
        [   73,   112,  8375,  7197],
        [   74,    59, 22394,    98],
        [  104, 24096, 20172,   153],
        [ 9484,    43,    73,  8563]])

Targets:
 tensor([[   44,    62,    69,    72],
        [   74,    13,    74,    59],
        [   72,    67,  1092,   117],
        [   68,    59,    73,    73],
        [  112,  8375,  7197,    74],
        [   59, 22394,    98,   104],
        [24096, 20172,   153,  9484],
        [   43,    73,  8563, 22394]])


## Attention

In [42]:
class SelfAttentionLayer(nn.Module):
    def __init__(self, d_in=EMB_DIM, d_out=EMB_DIM, qkv_bias=QKV_BIAS):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        context_vec = attn_weights @ values

        return context_vec


class CausalAttentionLayer(nn.Module):
    def __init__(
        self,
        d_in=EMB_DIM,
        d_out=EMB_DIM,
        context_len=CONTEXT_LEN,
        dropout=DROPOUT,
        qkv_bias=QKV_BIAS,
    ):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):
        _, num_tokens, _ = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec


class MultiHeadAttentionLayer(nn.Module):
    def __init__(
        self,
        d_in=EMB_DIM,
        d_out=EMB_DIM,
        context_len=CONTEXT_LEN,
        dropout=DROPOUT,
        num_heads=N_HEADS,
        qkv_bias=QKV_BIAS,
    ):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisble by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, _ = x.shape
        queries = (
            self.W_query(x)
            .view(b, num_tokens, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )
        keys = (
            self.W_key(x)
            .view(b, num_tokens, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )
        values = (
            self.W_value(x)
            .view(b, num_tokens, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (
            (attn_weights @ values)
            .transpose(1, 2)
            .contiguous()
            .view(b, num_tokens, self.d_out)
        )
        return self.out_proj(context_vec)

In [33]:
if __name__ == "__main__":
    ca = MultiHeadAttentionLayer(2, 6, 4, 0.0, 1)
    cvs = ca(torch.rand(5, 3, 2))
    print(cvs)
    print(cvs.shape)

tensor([[[-0.4399,  0.4598, -0.0233, -0.3491, -0.6184, -0.0781],
         [-0.3357,  0.2817, -0.0832, -0.2424, -0.4430,  0.0990],
         [-0.3091,  0.2365, -0.0983, -0.2143, -0.3990,  0.1451]],

        [[-0.3903,  0.4375, -0.0099, -0.1786, -0.6723,  0.1110],
         [-0.4221,  0.4732, -0.0041, -0.2469, -0.6848,  0.0256],
         [-0.4255,  0.4646, -0.0118, -0.2780, -0.6589, -0.0043]],

        [[-0.2861,  0.1794, -0.1235, -0.2247, -0.3212,  0.1544],
         [-0.3148,  0.2655, -0.0821, -0.1833, -0.4509,  0.1676],
         [-0.3546,  0.3535, -0.0459, -0.1861, -0.5616,  0.1331]],

        [[-0.3730,  0.3797, -0.0388, -0.2151, -0.5810,  0.0929],
         [-0.3967,  0.4306, -0.0182, -0.2195, -0.6438,  0.0700],
         [-0.4111,  0.4517, -0.0123, -0.2410, -0.6602,  0.0396]],

        [[-0.2573,  0.1687, -0.1142, -0.1214, -0.3575,  0.2680],
         [-0.3340,  0.2927, -0.0749, -0.2138, -0.4708,  0.1255],
         [-0.3316,  0.2861, -0.0779, -0.2160, -0.4613,  0.1254]]],
       grad_fn=

## MLP

In [21]:
class LayerNorm(nn.Module):
    def __init__(self, dim=EMB_DIM):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(dim))
        self.shift = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return (
            0.5
            * x
            * (
                1
                + torch.tanh(
                    torch.sqrt(torch.tensor(2.0 / torch.pi))
                    * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )


class FeedForward(nn.Module):
    def __init__(self, dim=EMB_DIM):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            GELU(),
            nn.Linear(4 * dim, dim),
        )

    def forward(self, x):
        return self.layers(x)


## Transformer

In [22]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = LayerNorm()
        self.att = MultiHeadAttentionLayer()
        self.ff = FeedForward()
        self.norm2 = LayerNorm()
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.dropout(x)
        x += shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.dropout(x)
        x += shortcut
        return x

In [24]:
if __name__ == "__main__":
    x = torch.rand(2, 4, EMB_DIM)
    block = TransformerBlock()
    output = block(x)

    print("Input shape:", x.shape)
    print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 1024])
Output shape: torch.Size([2, 4, 1024])


## GPT model

In [43]:
class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LEN, EMB_DIM)
        self.drop_emb = nn.Dropout(DROPOUT)
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock() for _ in range(N_LAYERS)]
        )
        self.final_norm = LayerNorm()
        self.out_head = nn.Linear(EMB_DIM, VOCAB_SIZE, bias=False)
        print("Model initialized")

    def forward(self, idx):
        _, seq_len = idx.shape
        tok_emb = self.tok_emb(idx)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=idx.device))

        x = self.drop_emb(tok_emb + pos_emb)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)  # logits

    def generate(self, idx, max_tokens, context_len=CONTEXT_LEN):
        res = []
        for t in idx:
            for i in range(t[0].shape[0], 0, -1):
                if t[0, i - 1] != 0:
                    res.append(t[:, :i])
                    break

        for b in range(len(res)):
            for _ in range(max_tokens):
                idx_cond = res[b][:, -context_len:]
                with torch.no_grad():
                    logits = self(idx_cond)

                logits = logits[:, -1, :]
                probas = torch.softmax(logits, dim=-1)
                idx_next = torch.argmax(probas, dim=-1, keepdim=True)
                res[b] = torch.cat((res[b], idx_next), dim=1)

        return res

In [36]:
if __name__ == "__main__":
    tokenizer = BPETokenizer()
    encoded = tokenizer.encode("Hello, I am").unsqueeze(0).unsqueeze(0)
    print("Encoded:", encoded)

    model = GPTModel()
    model.eval()
    torch.set_printoptions(sci_mode=False)
    for out in model.generate(encoded, 6):
        print("Output:", out)
        print("Output length:", len(out[0]))
        print("Decoded:", tokenizer.decode(out.squeeze(0)))

Encoded: tensor([[[ 33,  59,  66,  66,  69,  12, 167, 483]]])
Model initialized
Output: tensor([[   33,    59,    66,    66,    69,    12,   167,   483, 27847,  8751,
         44432, 38584, 32338, 17591]])
Output length: 14
Decoded: Hello, I am pedest BA Fusarium proteasom Attention thousand


## Pre-training

In [37]:
def calc_loss_batch(input_batch, target_batch, model, device=DEVICE):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    return F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())  # loss


def calc_loss_loader(data_loader, model, device=DEVICE, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        print(i)
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break

    return total_loss / num_batches

In [ ]:
if __name__ == "__main__":
    tokenizer = BPETokenizer()
    sample_data = load_sample_data()
    full = "<|endoftext|>".join(sample_data[:1000])
    split_idx = int(TRAIN_RATIO * len(full))

    train_data, val_data = full[:split_idx], full[split_idx:]
    train_loader = create_dataloader(train_data, tokenizer=tokenizer, batch_size=2)
    val_loader = create_dataloader(val_data, tokenizer=tokenizer, batch_size=2)
    print(f"Train loader - {len(train_loader)} batches:")
    print(f"Validation loader - {len(val_loader)} batches")

    model = GPTModel()
    model.to(DEVICE, dtype=MODEL_DTYPE)
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model)
        val_loss = calc_loss_loader(val_loader, model)
    print("Training loss:", train_loss)
    print("Validation loss:", val_loss)

Train loader - 885 batches:
Validation loader - 97 batches
Model initialized
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
2